🌾 Andhra Pradesh Multi-Commodity 3-Day Price Prediction Engine
### 18 Commodities | Data.gov.in API | Global XGBoost | 70/20/10 Split | Quantile Risk Bands

This notebook fetches **live data** from the Data.gov.in API for **Andhra Pradesh** markets, trains a **Global XGBoost** model, and generates **recursive 3-day price forecasts** with risk bands.

**Supported Commodities (18):**

| Category | Commodities |
|----------|-------------|
| **Grains & Cereals** | Paddy(Common), Maize |
| **Pulses & Oilseeds** | Groundnut, Bengal Gram(Gram)(Whole), Black Gram(Urd Beans)(Whole), Castor Seed, Red gram/Arhar/Tur(whole), Cotton |
| **Vegetables** | Tomato, Dry Chillies, Chili Red, Turmeric, Lemon, Lime |
| **Fruits** | Banana, Mango, Tamarind Fruit |
| **Processed** | Gur(Jaggery) |

**Model Split:** 70% Train · 20% Validation (hyperparameter tuning) · 10% Test (final accuracy)

## Step 1: Setup & Commodity Selection

**⬇️ Change `SELECTED_COMMODITY` below** to any of the 18 supported commodities.

In [ ]:
import urllib.request
import urllib.parse
import json
import pandas as pd
import numpy as np
import time
import warnings
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')

# ============================================================
# USER CONFIG: SELECT YOUR COMMODITY HERE
# ============================================================
SELECTED_COMMODITY = "Paddy(Common)"  # <-- CHANGE THIS

# All 18 viable commodities (200+ records, 3+ markets in AP)
VALID_COMMODITIES = [
    "Paddy(Common)", "Tomato", "Banana", "Maize", "Groundnut",
    "Dry Chillies", "Mango", "Gur(Jaggery)", "Lemon", "Cotton",
    "Turmeric", "Bengal Gram(Gram)(Whole)", "Chili Red",
    "Black Gram(Urd Beans)(Whole)", "Castor Seed", "Tamarind Fruit",
    "Lime", "Red gram/Arhar/Tur(whole)"
]

# Number of top markets to use (by record count)
TOP_N_MARKETS = 10

# Target state
TARGET_STATE = "Andhra Pradesh"

# Split ratios
TRAIN_RATIO = 0.70
VAL_RATIO = 0.20
TEST_RATIO = 0.10

# API Configuration
API_KEY = "579b464db66ec23bdd000001a0a99e04a75a40666201931688acb738"
RESOURCE_ID = "35985678-0d79-46b4-9ed6-6f13308a1d24"
BASE_URL = f"https://api.data.gov.in/resource/{RESOURCE_ID}"

assert SELECTED_COMMODITY in VALID_COMMODITIES, (
    f"Invalid: '{SELECTED_COMMODITY}'. Choose from:\n" + "\n".join(f"  - {c}" for c in VALID_COMMODITIES)
)

print(f"🌾 Commodity:  {SELECTED_COMMODITY}")
print(f"📍 State:      {TARGET_STATE}")
print(f"🎯 Markets:    Top {TOP_N_MARKETS} (auto-selected)")
print(f"📊 Split:      {int(TRAIN_RATIO*100)}% Train / {int(VAL_RATIO*100)}% Validation / {int(TEST_RATIO*100)}% Test")
print(f"\nAll supported commodities:")
for i, c in enumerate(VALID_COMMODITIES, 1):
    marker = " ◀ SELECTED" if c == SELECTED_COMMODITY else ""
    print(f"  {i:2d}. {c}{marker}")

## Step 2: Fetch Data from Data.gov.in API

Paginates through the API with **per-page state filtering** to guard against the data.gov.in overflow bug.

In [ ]:
def fetch_mandi_data(state, commodity):
    all_records = []
    offset = 0
    limit = 1000
    total = None

    print(f"Fetching {commodity} data for {state}...")

    while True:
        params = {
            "api-key": API_KEY, "format": "json",
            "limit": limit, "offset": offset,
            "filters[state]": state, "filters[commodity]": commodity
        }
        url = f"{BASE_URL}?{urllib.parse.urlencode(params)}"
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})

        try:
            with urllib.request.urlopen(req, timeout=15) as response:
                data = json.loads(response.read().decode('utf-8'))
                records = data.get("records", [])

                if total is None:
                    total = int(data.get("total", 0))
                    print(f"API reports {total} total records")

                if not records:
                    break

                # CRITICAL: Filter each page - API leaks other states past offset boundary
                valid = [r for r in records if r.get('State', '') == state]

                if len(valid) == 0:
                    print(f"[STOP] API overflow at offset {offset}. Stopping.")
                    break

                dropped = len(records) - len(valid)
                if dropped > 0:
                    print(f"  [FILTER] Dropped {dropped} non-{state} records at offset {offset}")

                all_records.extend(valid)
                offset += limit
                if offset >= total:
                    break
                time.sleep(0.2)

        except Exception as e:
            print(f"Error at offset {offset}: {e}")
            time.sleep(1.0)
            try:
                with urllib.request.urlopen(req, timeout=15) as response:
                    data = json.loads(response.read().decode('utf-8'))
                    recs = data.get("records", [])
                    valid = [r for r in recs if r.get('State','') == state]
                    all_records.extend(valid)
                    offset += limit
            except:
                break

    df = pd.DataFrame(all_records)
    if 'State' in df.columns and not df.empty:
        before = len(df)
        df = df[df['State'] == state].reset_index(drop=True)
        if len(df) < before:
            print(f"[SAFETY] Removed {before - len(df)} non-{state} rows")
    return df

raw_df = fetch_mandi_data(TARGET_STATE, SELECTED_COMMODITY)
print(f"\n✅ Total {TARGET_STATE} {SELECTED_COMMODITY} records: {len(raw_df)}")

## Step 3: Data Cleaning & Top-10 Market Selection

Parses dates, cleans names, selects **top 10 markets**, and filters to 2018+ for current price regimes.

In [ ]:
if raw_df.empty:
    raise ValueError(f"No data for {SELECTED_COMMODITY} in {TARGET_STATE}.")

df = raw_df.copy()
df['date'] = pd.to_datetime(df['Arrival_Date'], format='%d/%m/%Y', errors='coerce')
df['Market'] = df['Market'].astype(str).str.replace(' APMC', '').str.strip()
df['weighted_avg_modal_price'] = pd.to_numeric(df['Modal_Price'], errors='coerce')
df['min_price'] = pd.to_numeric(df['Min_Price'], errors='coerce')
df['max_price'] = pd.to_numeric(df['Max_Price'], errors='coerce')

if 'State' in df.columns:
    df = df[df['State'] == TARGET_STATE]

df = df.dropna(subset=['date', 'weighted_avg_modal_price'])
df = df[df['date'] >= '2018-01-01'].sort_values(['Market', 'date']).reset_index(drop=True)

# Select top N markets
market_counts = df['Market'].value_counts()
available_n = min(TOP_N_MARKETS, len(market_counts))
top_markets = market_counts.head(available_n).index.tolist()
df = df[df['Market'].isin(top_markets)].reset_index(drop=True)

print(f"✅ {SELECTED_COMMODITY}: {len(df)} rows across {available_n} top AP markets (2018+)")
print()
print(f"{'Top Markets':^60}")
print("=" * 60)
for mkt in top_markets:
    cnt = market_counts[mkt]
    dr = df[df['Market'] == mkt]['date']
    print(f"  {mkt:35s} {cnt:4d} rows  ({dr.min().strftime('%Y-%m-%d')} to {dr.max().strftime('%Y-%m-%d')})")

## Step 4: Feature Engineering

Price lags, rolling momentum, calendar features, min/max lags, and one-hot encoded markets.

In [ ]:
def create_features(df):
    df = df.sort_values(['Market', 'date']).copy()
    for lag in [1, 2, 3, 7]:
        df[f'lag_{lag}'] = df.groupby('Market')['weighted_avg_modal_price'].shift(lag)
    df['min_lag_1'] = df.groupby('Market')['min_price'].shift(1)
    df['max_lag_1'] = df.groupby('Market')['max_price'].shift(1)
    df['rolling_mean_3'] = df.groupby('Market')['lag_1'].transform(lambda x: x.rolling(3).mean())
    df['rolling_mean_7'] = df.groupby('Market')['lag_1'].transform(lambda x: x.rolling(7).mean())
    df['prev_spread'] = df['max_lag_1'] - df['min_lag_1']
    df['dayofweek'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df['day_of_year'] = df['date'].dt.dayofyear
    return df.dropna()

featured_df = create_features(df)
featured_df = pd.get_dummies(featured_df, columns=['Market'], prefix='mkt', drop_first=False)
print(f"✅ Features created: {len(featured_df)} rows, {len(featured_df.columns)} columns")

## Step 5: Train Global XGBoost with 70/20/10 Split

| Split | Purpose | Usage |
|-------|---------|-------|
| **70% Train** | Model fitting | XGBoost learns patterns |
| **20% Validation** | Hyperparameter tuning | Early stopping & model selection |
| **10% Test** | Final accuracy | **Unseen data** — reported accuracy |

The **test accuracy** is the only number that matters — it's on data the model has **never seen** during training or tuning.

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error

# Define Features
exclude_cols = ['date', 'weighted_avg_modal_price', 'min_price', 'max_price',
                'District', 'State', 'Commodity', 'Variety', 'Grade',
                'Arrival_Date', 'Market', 'Modal_Price', 'Min_Price', 'Max_Price',
                'Commodity_Code']
FEATURES = [c for c in featured_df.columns if c not in exclude_cols and featured_df[c].dtype != 'object']

# ── 70/20/10 Chronological Split ──
sorted_dates = featured_df['date'].sort_values()
train_cutoff = sorted_dates.quantile(TRAIN_RATIO)
val_cutoff = sorted_dates.quantile(TRAIN_RATIO + VAL_RATIO)

train_df = featured_df[featured_df['date'] <= train_cutoff]
val_df = featured_df[(featured_df['date'] > train_cutoff) & (featured_df['date'] <= val_cutoff)]
test_df = featured_df[featured_df['date'] > val_cutoff]

X_train, y_train = train_df[FEATURES], train_df['weighted_avg_modal_price']
X_val, y_val = val_df[FEATURES], val_df['weighted_avg_modal_price']
X_test, y_test = test_df[FEATURES], test_df['weighted_avg_modal_price']

print(f"Train:      {len(X_train):5d} rows  (up to {train_cutoff.strftime('%Y-%m-%d')})")
print(f"Validation: {len(X_val):5d} rows  ({train_cutoff.strftime('%Y-%m-%d')} to {val_cutoff.strftime('%Y-%m-%d')})")
print(f"Test:       {len(X_test):5d} rows  (after {val_cutoff.strftime('%Y-%m-%d')})")

# ── Train Models ──
params = {'max_depth': 5, 'learning_rate': 0.05, 'n_estimators': 500, 'random_state': 42,
          'early_stopping_rounds': 20}
models = {}
val_mae = 0
use_quantile = False

try:
    print("\nAttempting Quantile Regression...")
    models['p50'] = xgb.XGBRegressor(objective='reg:quantile', quantile_alpha=0.5,
                                      max_depth=5, learning_rate=0.05, n_estimators=500,
                                      random_state=42, early_stopping_rounds=20)
    models['p10'] = xgb.XGBRegressor(objective='reg:quantile', quantile_alpha=0.1,
                                      max_depth=5, learning_rate=0.05, n_estimators=500,
                                      random_state=42, early_stopping_rounds=20)
    models['p90'] = xgb.XGBRegressor(objective='reg:quantile', quantile_alpha=0.9,
                                      max_depth=5, learning_rate=0.05, n_estimators=500,
                                      random_state=42, early_stopping_rounds=20)

    # Use validation set for early stopping
    models['p50'].fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    models['p10'].fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    models['p90'].fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    use_quantile = True
    print(f"✅ Quantile models trained (P50 best iter: {models['p50'].best_iteration})")

except Exception as e:
    print(f"[FALLBACK] Quantile failed ({e}). Using squared error.")
    models['p50'] = xgb.XGBRegressor(objective='reg:squarederror',
                                      max_depth=5, learning_rate=0.05, n_estimators=500,
                                      random_state=42, early_stopping_rounds=20)
    models['p50'].fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    val_preds = models['p50'].predict(X_val)
    val_mae = np.mean(np.abs(y_val - val_preds))

# ── Evaluate on ALL THREE splits ──
def evaluate(X, y, label):
    pred = models['p50'].predict(X)
    mape = mean_absolute_percentage_error(y, pred) * 100
    mae = mean_absolute_error(y, pred)
    acc = max(0, 100 - mape)
    return {'Split': label, 'Rows': len(y), 'MAPE (%)': round(mape, 2),
            'MAE (Rs)': round(mae, 2), 'Accuracy (%)': round(acc, 2)}

# Baseline (Naive Lag-1)
baseline_test_mape = mean_absolute_percentage_error(y_test, X_test['lag_1']) * 100

results = [
    evaluate(X_train, y_train, 'Train (70%)'),
    evaluate(X_val, y_val, 'Validation (20%)'),
    evaluate(X_test, y_test, 'TEST (10%) ⭐'),
]

print()
print("=" * 75)
print(f"  MODEL PERFORMANCE: {SELECTED_COMMODITY} ({TARGET_STATE})")
print("=" * 75)
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print()
print(f"Naive Baseline (Lag-1) Test MAPE: {baseline_test_mape:.2f}%")
print()

test_acc = results[2]['Accuracy (%)']
test_mape = results[2]['MAPE (%)']
gen_gap = results[2]['MAPE (%)'] - results[0]['MAPE (%)']

if test_acc >= 90:
    grade = "🟢 EXCELLENT"
elif test_acc >= 80:
    grade = "🟡 GOOD"
elif test_acc >= 70:
    grade = "🟠 FAIR"
else:
    grade = "🔴 POOR"

print(f"⭐ FINAL TEST ACCURACY: {test_acc:.2f}% (MAPE: {test_mape:.2f}%) {grade}")
print(f"   Generalization Gap: {gen_gap:+.2f}% (Train→Test)")
if test_mape < baseline_test_mape:
    print(f"   ✅ ML beats baseline by {baseline_test_mape - test_mape:.2f}% MAPE")
else:
    print(f"   ⚠️ Baseline beats ML by {test_mape - baseline_test_mape:.2f}% MAPE")

## Step 6: 3-Day Forecast Engine with Actual Dates

Generates **recursive 3-day forecasts** showing exact prediction dates, risk bands, and trend indicators.

In [ ]:
def predict_3day(market_name, forecast_base_date=None):
    mkt_col = f'mkt_{market_name}'
    if mkt_col not in featured_df.columns:
        return None, f"Market '{market_name}' not found."

    mkt_hist = featured_df[featured_df[mkt_col] == 1].sort_values('date')
    if mkt_hist.empty:
        return None, f"No data for {market_name}."

    latest_row = mkt_hist.iloc[-1].copy()
    last_data_date = mkt_hist['date'].max()
    last_price = float(latest_row['weighted_avg_modal_price'])

    if forecast_base_date is None:
        forecast_base_date = max(pd.Timestamp(datetime.now().date()), last_data_date)

    predictions = []
    for day in range(1, 4):
        pred_date = forecast_base_date + timedelta(days=day)
        X_live = pd.DataFrame([latest_row])[FEATURES]
        p50 = float(models['p50'].predict(X_live)[0])

        if use_quantile:
            p10 = float(models['p10'].predict(X_live)[0])
            p90 = float(models['p90'].predict(X_live)[0])
        else:
            p10 = p50 - (1.5 * val_mae)
            p90 = p50 + (1.5 * val_mae)

        p10 = min(p10, p50)
        p90 = max(p90, p50)

        change = p50 - last_price
        change_pct = (change / last_price) * 100 if last_price > 0 else 0

        if change_pct > 2:
            trend = '📈 BULLISH'
        elif change_pct < -2:
            trend = '📉 BEARISH'
        else:
            trend = '➡️ STABLE'

        predictions.append({
            'Forecast Date': pred_date.strftime('%Y-%m-%d (%A)'),
            'Horizon': f'Day +{day}',
            'Expected (Rs/Q)': round(p50, 2),
            'Worst Case (Rs/Q)': round(p10, 2),
            'Best Case (Rs/Q)': round(p90, 2),
            'Change vs Last': f'{change:+.2f} ({change_pct:+.1f}%)',
            'Trend': trend,
        })

        latest_row['lag_7'] = latest_row.get('lag_6', latest_row['lag_3'])
        latest_row['lag_3'] = latest_row['lag_2']
        latest_row['lag_2'] = latest_row['lag_1']
        latest_row['lag_1'] = p50
        latest_row['rolling_mean_3'] = (latest_row['lag_1'] + latest_row['lag_2'] + latest_row['lag_3']) / 3.0
        latest_row['rolling_mean_7'] = (latest_row['lag_1'] * 3 + latest_row['rolling_mean_3'] * 4) / 7.0

    meta = {
        'market': market_name,
        'last_data_date': last_data_date.strftime('%Y-%m-%d'),
        'last_price': last_price,
        'forecast_base': forecast_base_date.strftime('%Y-%m-%d'),
    }
    return pd.DataFrame(predictions), meta

print("✅ Forecast engine ready.")

## Step 7: Generate 3-Day Forecasts for All Top Markets

Forecasts for every selected market with date-stamped predictions and summary table.

In [ ]:
all_markets = sorted([c.replace('mkt_', '') for c in FEATURES if c.startswith('mkt_')])
today = datetime.now()

print("=" * 95)
print(f"  🌾 {SELECTED_COMMODITY.upper()} — AP TOP {len(all_markets)} MARKETS — 3-DAY FORECAST")
print(f"  Generated: {today.strftime('%Y-%m-%d %H:%M IST')}  |  Test Accuracy: {test_acc:.2f}%")
print("=" * 95)

summary_rows = []

for market in all_markets:
    result, meta = predict_3day(market)
    if result is None:
        print(f"\n❌ {market}: {meta}")
        continue

    staleness = (pd.Timestamp(today.date()) - pd.Timestamp(meta['last_data_date'])).days
    stale_warn = f"  ⚠️ STALE ({staleness}d old)" if staleness > 7 else ""

    print(f"\n🏪 {market}{stale_warn}")
    print(f"   Last Data: {meta['last_data_date']}  |  Last Price: Rs. {meta['last_price']:.2f}  |  Forecast Base: {meta['forecast_base']}")
    print(result.to_string(index=False))
    print("-" * 95)

    for _, row in result.iterrows():
        summary_rows.append({
            'Market': market,
            'Date': row['Forecast Date'],
            'Expected (Rs/Q)': row['Expected (Rs/Q)'],
            'Range [Low - High]': f"[{row['Worst Case (Rs/Q)']} - {row['Best Case (Rs/Q)']}]",
            'Trend': row['Trend'],
        })

print()
print("=" * 95)
print(f"  SUMMARY: {SELECTED_COMMODITY} — ALL MARKETS × 3 DAYS")
print("=" * 95)
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

## Step 8: JSON API Output

Structured JSON for backend/dashboard integration.

In [ ]:
import json as json_lib

api_output = {
    'commodity': SELECTED_COMMODITY,
    'state': TARGET_STATE,
    'generated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'model': 'Global XGBoost (Quantile)' if use_quantile else 'Global XGBoost (Squared Error)',
    'split': '70/20/10 (Train/Val/Test)',
    'test_accuracy_pct': test_acc,
    'test_mape_pct': test_mape,
    'markets': {}
}

for market in all_markets:
    result, meta = predict_3day(market)
    if result is None:
        continue
    api_output['markets'][market] = {
        'last_data_date': meta['last_data_date'],
        'last_price': meta['last_price'],
        'forecasts': result.to_dict(orient='records')
    }

print(json_lib.dumps(api_output, indent=2, ensure_ascii=False))